# KenyaFreshRetail - Silver Schema Exploration
This notebook connects to the SQL Server instance and explores the tables in the silver schema of the KenyaFreshRetail database.

## 1. Import Required Libraries

In [1]:
import pyodbc
import pandas as pd
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings('ignore')

## 2. Establish SQL Server Connection

In [2]:
# SQL Server connection string - Update with your server details
# Using Windows Authentication (Trusted Connection)
server = 'localhost'  # Update with your SQL Server instance name
database = 'KenyaFreshRetail'

# Create connection string for pyodbc
connection_string = f'Driver={{ODBC Driver 17 for SQL Server}};Server={server};Database={database};Trusted_Connection=yes;'

# Test connection
try:
    conn = pyodbc.connect(connection_string)
    cursor = conn.cursor()
    print("✓ Successfully connected to SQL Server!")
    print(f"✓ Connected to database: {database}")
except Exception as e:
    print(f"✗ Connection failed: {str(e)}")
    print("Please update the server name and ensure SQL Server is running.")

✓ Successfully connected to SQL Server!
✓ Connected to database: KenyaFreshRetail


## 3. Query Silver Schema Tables

In [3]:
# Query to list all tables in the silver schema
query_silver_tables = """
SELECT TABLE_SCHEMA, TABLE_NAME
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'silver'
ORDER BY TABLE_NAME
"""

try:
    silver_tables = pd.read_sql(query_silver_tables, conn)
    print(f"Found {len(silver_tables)} table(s) in the silver schema:\n")
    print(silver_tables.to_string(index=False))
except Exception as e:
    print(f"Error querying tables: {str(e)}")

Found 11 table(s) in the silver schema:

TABLE_SCHEMA        TABLE_NAME
      silver competitor_stores
      silver       competitors
      silver               crm
      silver          economic
      silver      gis_counties
      silver     gis_locations
      silver                hr
      silver       load_errors
      silver               pos
      silver          products
      silver            stores


## 4. Inspect Table Structures

In [4]:
# Function to get table structure
def get_table_structure(table_name):
    query = f"""
    SELECT 
        COLUMN_NAME,
        DATA_TYPE,
        IS_NULLABLE,
        COLUMN_DEFAULT
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = 'silver' AND TABLE_NAME = '{table_name}'
    ORDER BY ORDINAL_POSITION
    """
    try:
        structure = pd.read_sql(query, conn)
        return structure
    except Exception as e:
        print(f"Error getting structure for {table_name}: {str(e)}")
        return None

# Display structure for each table
if not silver_tables.empty:
    for idx, row in silver_tables.iterrows():
        table_name = row['TABLE_NAME']
        print(f"\n{'='*60}")
        print(f"Table: [silver].[{table_name}]")
        print(f"{'='*60}")
        structure = get_table_structure(table_name)
        if structure is not None:
            print(structure.to_string(index=False))
else:
    print("No tables found in the silver schema.")


Table: [silver].[competitor_stores]
                  COLUMN_NAME DATA_TYPE IS_NULLABLE COLUMN_DEFAULT
                     store_id  nvarchar          NO           None
                competitor_id  nvarchar         YES           None
                       county  nvarchar         YES           None
                         town  nvarchar         YES           None
               store_size_sqm       int         YES           None
                 store_format  nvarchar         YES           None
                 opening_date      date         YES           None
estimated_monthly_revenue_kes   decimal         YES           None
    estimated_daily_customers       int         YES           None
               location_score       int         YES           None
            parking_available  nvarchar         YES           None
                 has_delivery  nvarchar         YES           None
                last_verified      date         YES           None
                  data_so

## 5. Display Sample Data

In [5]:
# Display sample data from each table
if not silver_tables.empty:
    for idx, row in silver_tables.iterrows():
        table_name = row['TABLE_NAME']
        print(f"\n{'='*60}")
        print(f"Sample Data from: [silver].[{table_name}]")
        print(f"{'='*60}")
        
        try:
            # Get row count
            count_query = f"SELECT COUNT(*) as [Row Count] FROM [silver].[{table_name}]"
            row_count = pd.read_sql(count_query, conn)
            print(row_count.to_string(index=False))
            
            # Get sample data (first 5 rows)
            sample_query = f"SELECT TOP 5 * FROM [silver].[{table_name}]"
            sample_data = pd.read_sql(sample_query, conn)
            print(f"\nFirst 5 rows:")
            print(sample_data.to_string(index=False))
            
        except Exception as e:
            print(f"Error retrieving sample data: {str(e)}")
else:
    print("No tables found in the silver schema.")


Sample Data from: [silver].[competitor_stores]
 Row Count
       203

First 5 rows:
          store_id competitor_id  county      town  store_size_sqm store_format opening_date  estimated_monthly_revenue_kes  estimated_daily_customers  location_score parking_available has_delivery last_verified    data_source          load_timestamp
COMP-CAR-STORE-001      COMP-CAR Nairobi Lavington             450  Hypermarket   2017-10-07                    22546053.50                        300               9           Limited           No    2025-07-13 Field_Research 2025-12-25 03:25:18.930
COMP-CAR-STORE-002      COMP-CAR Nairobi Lavington             734  Hypermarket   2022-12-18                    36706335.27                        300               9               Yes          Yes    2025-11-18 Field_Research 2025-12-25 03:25:18.930
COMP-CAR-STORE-003      COMP-CAR Mombasa     Nyali             632  Hypermarket   2019-12-07                    31617539.96                       3000            

## 6. Close Connection

In [6]:
# Close the connection when done
try:
    if conn:
        conn.close()
        print("✓ Connection closed successfully")
except Exception as e:
    print(f"Error closing connection: {str(e)}")

✓ Connection closed successfully
